In [8]:
############################################################################################
# SuperVisor 패턴 : 하나의 상위노드가 여러 작업자를 배치해서 각 작업의 결과를 모아서 최종 답변
############################################################################################
# 초안
# 비평
# 품질 점수
# 80점미만
# 재작성

#       Supervisor
#       Reasearcher
#       Writer
#       Critic
# faile         pass
# Writer        Finalizer

In [9]:
# 슈퍼바이저를 한단계 더 발전시켜서 에이전트 개념확장
#                     Supervisor
#                          │
#         ┌────────────────┼────────────────┐
#         │                │                │
#         ▼                ▼                ▼
#    ResearchAgent    WriterAgent    CriticAgent
#         │                │                │
#         └────────────────┼────────────────┘
#                          ▼
#                     Finalizer

#  Supervisor--> 오케스트레이터

In [10]:
import os
import re
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

# openai client
client = OpenAI()
# 임베딩 함수  허깅페이스 :This is a sentence-transformers model 찾아라.
st_ef =  embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='jhgan/ko-sroberta-multitask'
)
chroma_client = chromadb.PersistentClient()  #DB
collection = chroma_client.get_or_create_collection(   # table
    name = 'Supervisor_MultiAgent',
    embedding_function=st_ef
)

if collection.count() == 0:
    collection.add(
        ids=[
            "doc_langgraph",
            "doc_react",
            "doc_rag",
            "doc_supervisor",
        ],
        documents=[
            "LangGraph는 상태 기반 그래프로 역할별 노드를 연결한다.",
            "ReAct는 생각과 행동을 번갈아 수행하며 중간 도구를 사용할 수 있다.",
            "RAG는 벡터DB에서 찾은 근거를 답변 생성에 사용한다.",
            "Supervisor는 여러 에이전트의 순서와 책임을 조율한다.",
        ],
    )

class SupervisorState(TypedDict):
    task:str
    research_notes:str
    draft:str
    critique:str
    final_answer:str

def supervisor(state:SupervisorState):
    return state

def researcher(state:SupervisorState):
    result = collection.query(
        query_texts=[state['task']],
        n_results=3
    )
    context = '\n'.join(result['documents'][0])
    response = client.chat.completions.create(
        model='gpt-5.4-nano',
        messages=[
            {'role':'system','content':'너는 Reasearch Agent이고 주어진 근거를 요약합니다.'},
            {'role':'user','content':f"작업:{state['task']}\n\n근거:{context}"}
        ]
    )    
    return {
        'research_notes': response.choices[0].message.content.strip()
    }

# -------------------------
# Writer Agent
# -------------------------

def writer(state: SupervisorState):

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "당신은 Writer Agent이다."
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

조사 내용:
{state['research_notes']}

설명문 초안을 작성하라.
"""
            },
        ],
    )

    return {
        "draft": response.choices[0].message.content.strip()
    }


# -------------------------
# Critic Agent
# -------------------------

def critic(state: SupervisorState):

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "당신은 Critic Agent이다."
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

초안:
{state['draft']}

부족한 점과 개선점을 평가하라.
"""
            },
        ],
    )

    return {
        "critique":response.choices[0].message.content.strip()
    }


# -------------------------
# Finalizer Agent
# -------------------------

def finalizer(state: SupervisorState):

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "최종 답변 작성 Agent"
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

조사:
{state['research_notes']}

초안:
{state['draft']}

비평:
{state['critique']}

비평을 반영하여 최종 답변을 작성하라.
"""
            },
        ],
    )

    return {
        "final_answer":
        response.choices[0].message.content.strip()
    }


graph = StateGraph(    SupervisorState)
graph.add_node(    "supervisor",    supervisor)
graph.add_node(    "researcher",    researcher)
graph.add_node(    "writer",    writer)
graph.add_node(    "critic",    critic)
graph.add_node(    "finalizer",    finalizer)
graph.add_edge(    START,    "supervisor")
graph.add_edge(    "supervisor",    "researcher")
graph.add_edge(    "researcher",    "writer")
graph.add_edge(    "writer",    "critic")
graph.add_edge(    "critic",    "finalizer")
graph.add_edge(    "finalizer",    END)
app = graph.compile()
result = app.invoke(
    {
        "task":
        "LangGraph Supervisor 패턴을 설명하라",
        "research_notes": "",
        "draft": "",
        "critique": "",
        "final_answer": "",
    }
)

print("\n=== FINAL ===")
print(result["final_answer"])

print("\n=== RESEARCH ===")
print(result["research_notes"])

print("\n=== CRITIQUE ===")
print(result["critique"])


=== FINAL ===
## LangGraph Supervisor 패턴 설명

**LangGraph Supervisor 패턴**은 LangGraph의 **상태(state) 기반 그래프** 위에, 여러 **worker(역할별 에이전트) 노드**를 두고, 이를 **Supervisor(컨트롤러) 노드가 상태를 근거로 라우팅(다음 실행 노드 선택)** 하며 **반복/종료**까지 관리하는 설계 방식이다.  
핵심은 “여러 에이전트가 각자 일하고, Supervisor가 `state`를 보고 **다음에 누구를 실행할지(`next`)** 결정한다”는 점이다.

---

## 1) LangGraph의 구조 관점: 상태가 ‘연결고리’
LangGraph에서는 노드 간에 단순히 “다음 노드로 점프”하는 것이 아니라, **공유 상태 스키마**를 중심으로 흐름이 구성된다.

- 그래프의 입력/출력/중간 산출물은 **state 필드**로 누적·갱신된다.
- 각 worker 노드는 보통
  - **(입력)** state의 일부를 읽고
  - **(처리)** 자기 역할을 수행한 뒤
  - **(출력)** state의 특정 필드(예: `draft`, `retrieved_context`, `needs_revision`)를 갱신한다.

따라서 Supervisor는 “작업을 분담”하는 역할뿐 아니라, **state 변화에 따라 다음 단계로 전환**시키는 역할까지 수행한다.

---

## 2) Supervisor의 역할: `next`를 결정하고 루프를 만든다
Supervisor 패턴에서 Supervisor는 보통 다음을 담당한다.

1. 현재 `state`를 읽는다. (예: 계획이 있는지, 근거가 충분한지, 품질이 낮은지)
2. 그 결과로 **다음에 호출할 worker를 선택**한다.
3. 선택 결과를 `state.next`(또는 그래프 라우팅에 쓰는 동일한 필드/리턴값)로 반영한다.
4. worker가 상태를 업데이트하면 Supervisor가 **다시** 라우팅한다.
5. 종료 조건이 충족되